# Evaluation audit (September 2026) — part B
### Adaptation budget sweep (Figure 2 of the paper)

Does giving the mGPT-1.3B + QLoRA baseline more Uzbek data close the gap to uzbek-gpt-103m? This notebook fine-tunes it at 0, 1M and 10M adaptation tokens and scores each on the deduplicated held-out set.

| Step | What it does | Script |
|---|---|---|
| 1 | Rebuilds the deduplicated held-out set in this session (identical to part A) | `02_clean_eval_and_score.py` |
| 2 | Runs the budget sweep and draws Figure 2 | `04_adaptation_sweep.py` |

**To run on Kaggle:** GPU T4, internet on, input `uzbek-fineweb2-tokens-16k`. About 5 hours, most of it the 10M-token training run. Every stage caches its result in `/kaggle/working`, so a timeout does not restart finished work.

**About the outputs.** This session's own outputs, with download progress bars and library loading warnings removed. Same text in `logs/`. Step 1 here is the third session of the evaluation script; its deterministic results match part A exactly.

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate safetensors huggingface_hub matplotlib

## Step 1 — Rebuild the clean held-out set

Kaggle sessions do not keep files, so the held-out set is rebuilt here before the sweep. The result is the same 474 documents, and every deterministic score is identical to part A.

In [ ]:
"""
================================================================================
UZBEK-GPT PAPER — VERIFICATION RUN v3  (DEDUPLICATED EVAL SET)
================================================================================
Why v3 exists:

  v1  eval set was the literal opening of train.bin. Void.
  v2  eval set decoded from val.bin — but 6/16 probes still turned up in
      train.bin, at scattered positions (252M, 929M, 833M, ...). That is not a
      split error. FineWeb-2 itself contains the same passages in both halves.
      So val.bin is partly contaminated too, and no choice of slice fixes it.
  v3  filters DOCUMENT BY DOCUMENT. Builds a fingerprint index of every 24-token
      window in train.bin, then keeps only val documents with zero fingerprint
      overlap. What survives is text the from-scratch model provably never saw.

CONSEQUENCE YOU SHOULD ABSORB
  Your model's reported validation loss of 3.059 was measured on val.bin, which
  we now know is partly duplicated from train.bin. That number is optimistic too.
  This is a property of the corpus, not a mistake you made — but the paper has to
  say the eval set was deduplicated against the training split, because from now
  on it will have been.

STAGES
  1  fingerprint index of train.bin            (~3 min, one pass)
  2  walk val.bin, drop duplicated documents, build the clean eval set
  3  verify with exact search — must be 0/16
  4  rebuild the old contaminated set, for the memorisation delta
  5  adaptation pool + text-level leak check
  6  QLoRA fine-tune mGPT-1.3B, ctx 512, all-linear, ~1M tokens
  7  Protocol A (512-token chunks): 3 models on clean + from-scratch on old
  8  Protocol B (byte-aligned spans): 3 models on clean
  9  paired bootstrap
  10 RESULTS

RUNTIME ~70-90 min on a T4. Restartable; every stage caches to /kaggle/working.

FIRST, IN A SEPARATE CELL:
    !pip install -q transformers datasets peft bitsandbytes accelerate safetensors huggingface_hub
Accelerator: GPU T4. Internet: ON. Add Input: uzbek-fineweb2-tokens-16k.
================================================================================
"""

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, sys, math, json, csv, glob, time
import numpy as np
import torch

LN2  = math.log(2.0)
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

# ------------------------------- config -------------------------------------
BASE          = "ai-forever/mGPT"
MY_MODEL_REPO = "IslombekT/uzbek-gpt-103m"
MY_TOK_REPO   = "IslombekT/uzbek-bpe-16k"
ADAPTER_DIR   = f"{WORK}/mgpt-uz-qlora-ctx512-dedup"

EOT_ID        = 0
KGRAM         = 24          # fingerprint window, ~18 Uzbek words
KEEP_MASK     = 31          # keep 1/32 of windows in the index
MAX_DUP_FRAC  = 0.0         # a document is rejected on ANY fingerprint match
MIN_DOC_WORDS = 30          # skip stubs, too short to fingerprint meaningfully
EVAL_WORDS    = 200_000

ADAPT_TOKENS  = 1_000_000
CTX_TRAIN     = 512
PER_DEV, ACC  = 2, 2
LR            = 2e-4

CHUNK_TOKENS  = 512
SPAN_BYTES    = 800
CONTEXT_BYTES = 300
MAX_TOKENS    = 512

N_PROBES      = 16
PROBE_TOK     = 32
RESAMPLES     = 10_000
SEED          = 20260913

assert torch.cuda.is_available(), "Enable a GPU: Settings -> Accelerator -> GPU T4."
DEV  = "cuda"
BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DT   = torch.bfloat16 if BF16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} | bf16: {BF16}")

def free(): gc.collect(); torch.cuda.empty_cache()
def stage(n, name): print(f"\n{'='*72}\nSTAGE {n} — {name}\n{'='*72}")

R = {"config": {"kgram": KGRAM, "eval_words": EVAL_WORDS, "adapt_tokens": ADAPT_TOKENS,
                "ctx_train": CTX_TRAIN, "span_bytes": SPAN_BYTES,
                "resamples": RESAMPLES, "seed": SEED}}

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, default_data_collator)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from torch.utils.data import Dataset

_B = np.uint64(1000003)

def kgram_hashes(arr, K=KGRAM, mask=KEEP_MASK, chunk=20_000_000):
    """Rolling polynomial hash of every K-gram, subsampled to 1/(mask+1)."""
    out, n = [], len(arr)
    if n < K:
        return np.zeros(0, dtype=np.uint64)
    for s in range(0, n - K + 1, chunk):
        e = min(s + chunk, n - K + 1); m = e - s
        h = np.zeros(m, dtype=np.uint64)
        for j in range(K):
            h = h * _B + arr[s+j : s+j+m].astype(np.uint64)
        out.append(h[(h & np.uint64(mask)) == 0])
    return np.concatenate(out) if out else np.zeros(0, dtype=np.uint64)

def find_sequence(hay, needle):
    L = len(needle)
    if L == 0 or len(hay) < L: return []
    cand = np.flatnonzero(hay[:len(hay)-L+1] == needle[0])
    for k in range(1, L):
        if cand.size == 0: break
        cand = cand[hay[cand+k] == needle[k]]
    return cand.tolist()

# =============== STAGE 1 — fingerprint index of train.bin ===================
stage(1, "fingerprint every 24-token window of train.bin")
cands = glob.glob("/kaggle/input/**/train.bin", recursive=True)
if not cands:
    raise SystemExit("train.bin not found. Add Input -> uzbek-fineweb2-tokens-16k")
TRAIN_BIN = cands[0]
VAL_BIN   = os.path.join(os.path.dirname(TRAIN_BIN), "val.bin")
train_ids = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_ids   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")
print(f"train.bin {len(train_ids):,} tokens | val.bin {len(val_ids):,} tokens")

IDX_FILE = f"{WORK}/train_fingerprints.npy"
if os.path.exists(IDX_FILE):
    IDX = np.load(IDX_FILE); print(f"reusing index: {len(IDX):,} fingerprints")
else:
    t0 = time.time()
    IDX = np.unique(kgram_hashes(np.asarray(train_ids)))
    np.save(IDX_FILE, IDX)
    print(f"built {len(IDX):,} fingerprints in {time.time()-t0:.0f}s ({IDX.nbytes/1e6:.0f} MB)")

def dup_fraction(tok_ids):
    h = kgram_hashes(np.asarray(tok_ids, dtype=np.uint16))
    if len(h) == 0: return 0.0, 0
    p = np.searchsorted(IDX, h); p[p >= len(IDX)] = 0
    return float((IDX[p] == h).mean()), len(h)

# ============ STAGE 2 — deduplicated eval set from val.bin ==================
stage(2, "build the eval set from val documents that are absent from train.bin")
tk = AutoTokenizer.from_pretrained(MY_TOK_REPO); tk.model_max_length = 10**9
CLEAN_FILE = f"{WORK}/uz_heldout_dedup.txt"

if os.path.exists(CLEAN_FILE):
    TEXT = open(CLEAN_FILE, encoding="utf-8").read()
    print(f"reusing {CLEAN_FILE}")
    R["docs_kept"] = R.get("docs_kept"); R["docs_rejected"] = R.get("docs_rejected")
else:
    kept, rejected, words, cur, i, shown = [], 0, 0, [], 0, 0
    while i < len(val_ids) and val_ids[i] != EOT_ID:   # start after a boundary
        i += 1
    i += 1
    while i < len(val_ids) and words < EVAL_WORDS:
        t = int(val_ids[i])
        if t == EOT_ID:
            if len(cur) >= MIN_DOC_WORDS:
                frac, n_fp = dup_fraction(cur)
                if n_fp > 0 and frac <= MAX_DUP_FRAC:
                    d = tk.decode(cur).strip()
                    if d:
                        kept.append(d); words += len(d.split())
                else:
                    rejected += 1
                    if shown < 3 and n_fp > 0 and frac > 0:
                        snip = tk.decode(cur[:40]).strip().replace("\n", " ")
                        print(f"  rejected ({frac:.0%} of windows in train): {snip[:110]}...")
                        shown += 1
            cur = []
        else:
            cur.append(t)
        i += 1
    TEXT = "\n".join(kept)
    open(CLEAN_FILE, "w", encoding="utf-8").write(TEXT)
    total = len(kept) + rejected
    print(f"\nkept {len(kept):,} documents, rejected {rejected:,} "
          f"({rejected/max(total,1):.1%} of val documents are duplicated in train)")
    R["docs_kept"], R["docs_rejected"] = len(kept), rejected

EVAL_BYTES = len(TEXT.encode("utf-8"))
print(f"eval set: {len(TEXT.split()):,} words, {EVAL_BYTES:,} bytes")
R["clean_eval_bytes"], R["clean_eval_words"] = EVAL_BYTES, len(TEXT.split())

# ==================== STAGE 3 — verify with exact search ====================
stage(3, "verify: exact search for eval passages inside train.bin")
eval_tok = np.asarray(tk(TEXT, add_special_tokens=False)["input_ids"], dtype=np.uint16)
pos = np.linspace(0, len(eval_tok)-PROBE_TOK-1, N_PROBES).astype(int)
hits = 0
for n, p in enumerate(pos, 1):
    h = find_sequence(train_ids, np.asarray(eval_tok[p:p+PROBE_TOK], dtype=np.uint16))
    hits += bool(h)
    print(f"  probe {n:>2}  eval tok {p:>9,}  " + (f"FOUND @{h[0]:,}" if h else "absent"))
print(f"\n{hits}/{N_PROBES} probes found in train.bin")
R["probes_in_train"] = f"{hits}/{N_PROBES}"
if hits > 0:
    raise SystemExit(f"ABORT — {hits}/{N_PROBES} still leaking. Lower KEEP_MASK to 15 "
                     f"(denser index) and rerun.")
print("eval set verified clean — proceeding")

# ============ STAGE 4 — the old contaminated set, for the delta =============
stage(4, "rebuild the OLD contaminated eval set (for the memorisation delta)")
OLD_FILE = f"{WORK}/uz_heldout.txt"
if os.path.exists(OLD_FILE):
    OLD_TEXT = open(OLD_FILE, encoding="utf-8").read(); print(f"reusing {OLD_FILE}")
else:
    ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn", split="train", streaming=True)
    docs, w = [], 0
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if not t: continue
        docs.append(t); w += len(t.split())
        if w >= EVAL_WORDS: break
    OLD_TEXT = "\n".join(docs); open(OLD_FILE, "w", encoding="utf-8").write(OLD_TEXT)
print(f"old eval set: {len(OLD_TEXT.split()):,} words")

# ================== STAGE 5 — adaptation pool + leak check ==================
stage(5, "adaptation pool")
POOL_FILE = f"{WORK}/adapt_dedup_{ADAPT_TOKENS}.npy"
tok_m = AutoTokenizer.from_pretrained(BASE)
if tok_m.pad_token is None: tok_m.pad_token = tok_m.eos_token
tok_m.model_max_length = 10**9

if os.path.exists(POOL_FILE):
    POOL = np.load(POOL_FILE); pool_text = ""
    print(f"reusing pool: {len(POOL)/1e6:.2f}M tokens")
else:
    need = int(ADAPT_TOKENS * 1.10)
    ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn", split="train", streaming=True)
    ids, chunks = [], []
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if not t: continue
        chunks.append(t)
        ids.extend(tok_m(t, add_special_tokens=False)["input_ids"] + [tok_m.eos_token_id])
        if len(ids) >= need: break
    POOL = np.asarray(ids[:need], dtype=np.int32); np.save(POOL_FILE, POOL)
    pool_text = "\n".join(chunks)
    print(f"tokenized {len(POOL)/1e6:.2f}M tokens from {len(chunks):,} docs")

if pool_text:
    probes = [TEXT[i:i+300] for i in np.linspace(0, max(len(TEXT)-400, 1), 12).astype(int)]
    leak = sum(1 for p in probes if p in pool_text)
    print(f"eval-text probes inside the adaptation pool: {leak}/{len(probes)}")
    R["adapt_pool_leak"] = f"{leak}/{len(probes)}"

# ======================= STAGE 6 — QLoRA fine-tune ==========================
stage(6, f"QLoRA mGPT-1.3B (ctx {CTX_TRAIN}, all-linear r=16, ~{ADAPT_TOKENS/1e6:.0f}M tokens)")
if os.path.exists(f"{ADAPTER_DIR}/adapter_model.safetensors"):
    print(f"reusing adapter: {ADAPTER_DIR}")
else:
    class Blocks(Dataset):
        def __init__(s, a, c): s.a, s.c, s.n = a, c, len(a)//c
        def __len__(s): return s.n
        def __getitem__(s, i):
            x = s.a[i*s.c:(i+1)*s.c].astype(np.int64)
            return {"input_ids": torch.from_numpy(x), "labels": torch.from_numpy(x.copy())}
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=DT, bnb_4bit_use_double_quant=True)
    m = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map={"": 0})
    m.config.use_cache = False
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    m = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                                     task_type="CAUSAL_LM", target_modules="all-linear"))
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"trainable: {trainable:,}"); R["trainable_params"] = trainable
    steps = ADAPT_TOKENS // (PER_DEV * ACC * CTX_TRAIN)
    print(f"steps: {steps} x {PER_DEV*ACC*CTX_TRAIN} = {steps*PER_DEV*ACC*CTX_TRAIN:,} tokens")
    args = TrainingArguments(
        output_dir=ADAPTER_DIR, per_device_train_batch_size=PER_DEV,
        gradient_accumulation_steps=ACC, max_steps=steps, learning_rate=LR,
        lr_scheduler_type="cosine", warmup_steps=max(10, steps//20),
        logging_steps=50, save_strategy="no", report_to=[],
        fp16=not BF16, bf16=BF16, gradient_checkpointing=True,
        optim="paged_adamw_8bit", seed=SEED)
    t0 = time.time()
    Trainer(model=m, args=args, train_dataset=Blocks(POOL, CTX_TRAIN),
            data_collator=default_data_collator).train()
    print(f"trained in {(time.time()-t0)/60:.1f} min")
    m.save_pretrained(ADAPTER_DIR); del m; free()

# ============================ model loaders =================================
def load_from_scratch():
    mp = hf_hub_download(MY_MODEL_REPO, "model.py"); sys.path.insert(0, os.path.dirname(mp))
    from model import GPT
    g = GPT(vocab_size=16384, n_embd=768, block_size=1024, num_heads=12, n_layers=12)
    g.load_state_dict(load_file(hf_hub_download(MY_MODEL_REPO, "model.safetensors")), strict=False)
    return g.eval().to(DEV)

def load_mgpt(adapter=None):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=DT, bnb_4bit_use_double_quant=True)
    mm = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                              device_map={"": 0}).eval()
    return PeftModel.from_pretrained(mm, adapter).eval() if adapter else mm

def logits_of(model, x):
    out = model(x)
    return out[0] if isinstance(out, tuple) else getattr(out, "logits", out)

# ========================= STAGE 7 — Protocol A =============================
stage(7, "Protocol A — 512-token chunks")

@torch.no_grad()
def protocol_a(tok, model, text, tag):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    n = len(ids) // CHUNK_TOKENS
    C = torch.tensor(ids[:n*CHUNK_TOKENS], dtype=torch.long).view(n, CHUNK_TOKENS)
    nats, byts, ntok = 0.0, 0, 0
    for i in range(n):
        x = C[i:i+1].to(DEV)
        lg = logits_of(model, x); sl, st = lg[:, :-1, :].float(), x[:, 1:]
        nats += torch.nn.functional.cross_entropy(
            sl.reshape(-1, sl.size(-1)), st.reshape(-1), reduction="sum").item()
        byts += len(tok.decode(st[0].tolist()).encode("utf-8")); ntok += st.numel()
    out = {"bpb": nats/(LN2*byts), "ce": nats/ntok, "chunks": n, "bytes": byts}
    print(f"  [{tag}] bpb={out['bpb']:.4f}  CE={out['ce']:.4f}  chunks={n}")
    return out

# ========================= STAGE 8 — Protocol B =============================
def build_spans(text, span_bytes):
    spans, i, n, k = [], 0, len(text), 0
    while i < n:
        j, b = i, 0
        while j < n and b < span_bytes:
            b += len(text[j].encode("utf-8")); j += 1
        spans.append((f"span_{k:05d}", i, j)); i, k = j, k+1
    if spans and len(text[spans[-1][1]:spans[-1][2]].encode("utf-8")) < span_bytes//2:
        spans.pop()
    return spans

SPANS = build_spans(TEXT, SPAN_BYTES)

@torch.no_grad()
def protocol_b(tok, model, tag):
    rows, over = [], 0
    for span_id, a, b in SPANS:
        n_bytes = len(TEXT[a:b].encode("utf-8"))
        c, got = a, 0
        while c > 0 and got < CONTEXT_BYTES:
            c -= 1; got += len(TEXT[c].encode("utf-8"))
        enc = tok(TEXT[c:b], return_offsets_mapping=True, add_special_tokens=False)
        ids, offs = enc["input_ids"], enc["offset_mapping"]
        first = next((k for k, (s, e) in enumerate(offs) if s >= a - c), len(ids))
        if len(ids) < 2: continue
        if len(ids) > MAX_TOKENS:
            over += 1; drop = len(ids) - MAX_TOKENS
            ids = ids[drop:]; first = max(0, first - drop)
        x = torch.tensor(ids, dtype=torch.long, device=DEV).unsqueeze(0)
        lg = logits_of(model, x)
        logits = lg[:, :-1, :].float(); target = x[:, 1:].clone()
        if first > 1: target[:, :first-1] = -100
        nats = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)), target.reshape(-1),
            ignore_index=-100, reduction="sum").item()
        rows.append((span_id, nats, n_bytes))
    path = f"{WORK}/dedup_chunks_{tag}.csv"
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh); w.writerow(["chunk_id","sum_nats","n_bytes"])
        w.writerows([(i, f"{v:.6f}", n) for i, v, n in rows])
    bpb = sum(r[1] for r in rows)/(LN2*sum(r[2] for r in rows))
    print(f"  [{tag}] bpb={bpb:.4f}  spans={len(rows)}" + (f"  ({over} truncated)" if over else ""))
    return bpb

A, B = {}, {}
g = load_from_scratch()
A["from_scratch_clean"] = protocol_a(tk, g, TEXT, "from_scratch / CLEAN")
A["from_scratch_old"]   = protocol_a(tk, g, OLD_TEXT, "from_scratch / OLD (contaminated)")
stage(8, f"Protocol B — {len(SPANS)} byte-aligned spans")
B["from_scratch"] = protocol_b(tk, g, "from_scratch")
del g; free()

base = load_mgpt()
A["mgpt_base"] = protocol_a(tok_m, base, TEXT, "mgpt_base / CLEAN")
B["mgpt_base"] = protocol_b(tok_m, base, "mgpt_base")
del base; free()

q = load_mgpt(adapter=ADAPTER_DIR)
A["mgpt_qlora"] = protocol_a(tok_m, q, TEXT, "mgpt_qlora / CLEAN")
B["mgpt_qlora"] = protocol_b(tok_m, q, "mgpt_qlora")
del q; free()

R["protocol_a"], R["protocol_b"] = A, B
DELTA = A["from_scratch_clean"]["bpb"] - A["from_scratch_old"]["bpb"]
R["contamination_delta_bpb"] = DELTA

# ======================= STAGE 9 — paired bootstrap =========================
stage(9, f"paired bootstrap, {RESAMPLES:,} resamples")

def load_csv(tag):
    d = {}
    with open(f"{WORK}/dedup_chunks_{tag}.csv", encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            d[r["chunk_id"]] = (float(r["sum_nats"]), float(r["n_bytes"]))
    return d

def paired(ta, tb):
    a, b = load_csv(ta), load_csv(tb)
    assert set(a) == set(b), "span sets differ"
    ids = sorted(a)
    na = np.array([a[i][0] for i in ids]); ba = np.array([a[i][1] for i in ids])
    nb = np.array([b[i][0] for i in ids]); bb = np.array([b[i][1] for i in ids])
    assert np.allclose(ba, bb), "paired spans differ in bytes"
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(ids), size=(RESAMPLES, len(ids)))
    d = (nb[idx].sum(1)/(LN2*bb[idx].sum(1))) - (na[idx].sum(1)/(LN2*ba[idx].sum(1)))
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"diff": nb.sum()/(LN2*bb.sum()) - na.sum()/(LN2*ba.sum()),
            "ci_lo": lo, "ci_hi": hi, "excludes_zero": bool(lo > 0 or hi < 0),
            "p_le_zero": float((d <= 0).mean()), "n_spans": len(ids)}

boots = {"vs_qlora": paired("from_scratch", "mgpt_qlora"),
         "vs_base":  paired("from_scratch", "mgpt_base")}
R["bootstrap"] = boots

# ============================ STAGE 10 — results ============================
stage(10, "RESULTS")
bq, bb_ = boots["vs_qlora"], boots["vs_base"]
print(f"""
eval set            val.bin documents with zero 24-token overlap with train.bin
                    {R['clean_eval_words']:,} words, {EVAL_BYTES:,} bytes
documents           kept {R.get('docs_kept','?')}, rejected {R.get('docs_rejected','?')} as duplicates
exact-search probes {R['probes_in_train']}  (0/{N_PROBES} required)
adaptation          {ADAPT_TOKENS:,} tokens, ctx {CTX_TRAIN}, all-linear r=16

MEMORISATION EFFECT (same model, same protocol, two eval sets)
  from-scratch on OLD   (trained on it)   {A['from_scratch_old']['bpb']:.4f} bpb
  from-scratch on CLEAN (never seen)      {A['from_scratch_clean']['bpb']:.4f} bpb
  memorisation was worth                  {DELTA:+.4f} bpb

PROTOCOL A — 512-token chunks, CLEAN eval set
  model                  bpb      per-token CE
  from-scratch 103M    {A['from_scratch_clean']['bpb']:.4f}      {A['from_scratch_clean']['ce']:.4f}
  mGPT-1.3B base       {A['mgpt_base']['bpb']:.4f}      {A['mgpt_base']['ce']:.4f}
  mGPT-1.3B + QLoRA    {A['mgpt_qlora']['bpb']:.4f}      {A['mgpt_qlora']['ce']:.4f}
  paper claimed        1.1050 / 1.1628 / 1.1214  (on contaminated text)

PROTOCOL B — {len(SPANS)} byte-aligned spans, CLEAN eval set
  from-scratch 103M    {B['from_scratch']:.4f}
  mGPT-1.3B base       {B['mgpt_base']:.4f}
  mGPT-1.3B + QLoRA    {B['mgpt_qlora']:.4f}

PAIRED BOOTSTRAP ({RESAMPLES:,} resamples, seed {SEED}; positive = from-scratch better)
  vs QLoRA baseline  {bq['diff']:+.4f} bpb  95% CI [{bq['ci_lo']:+.4f}, {bq['ci_hi']:+.4f}]  excludes zero: {'YES' if bq['excludes_zero'] else 'NO'}  P(<=0)={bq['p_le_zero']:.4f}
  vs zero-shot base  {bb_['diff']:+.4f} bpb  95% CI [{bb_['ci_lo']:+.4f}, {bb_['ci_hi']:+.4f}]  excludes zero: {'YES' if bb_['excludes_zero'] else 'NO'}  P(<=0)={bb_['p_le_zero']:.4f}
""")

json.dump(R, open(f"{WORK}/results_v3.json", "w"), indent=2, default=float)
print(f"saved -> {WORK}/results_v3.json, dedup_chunks_*.csv, {CLEAN_FILE}")


GPU: Tesla T4 | bf16: False

STAGE 1 — fingerprint every 24-token window of train.bin
train.bin 954,981,404 tokens | val.bin 106,109,045 tokens
built 27,405,976 fingerprints in 169s (219 MB)

STAGE 2 — build the eval set from val documents that are absent from train.bin
  rejected (71% of windows in train): 31 iyul 2011 Xalqaro kuzatuvchilarga ko‘ra, ikki yil oldin Urumchida yuz bergan zo‘ravonliklardan beri Xitoy u...
  rejected (6% of windows in train): - Подробности - Опубликовано: 18.07.2015 01:05 Insoniyatni doimo bir muammo o‘ylantirgani-o‘y...
  rejected (100% of windows in train): Bu mukofot har yili “Eng yaxshi badiiy film”, “Eng yaxshi televizionniy film”, “Eng yaxshi bolalar va o‘smirla...

kept 474 documents, rejected 306 (39.2% of val documents are duplicated in train)
eval set: 200,818 words, 1,646,167 bytes

STAGE 3 — verify: exact search for eval passages inside train.bin
  probe  1  eval tok         0  absent
  probe  2  eval tok    23,443  absent
  probe  3  eval tok 

## Step 2 — Adaptation budget sweep

The adaptation pool streams FineWeb-2 from the start (the training region), while the held-out set comes from `val.bin`. The script checks the pool text against the held-out text anyway, and stops if they overlap.

An earlier version of this sweep, run on the contaminated held-out set, reported 1.163 → 1.121 → 1.120 and concluded that adaptation plateaus. On clean data it does not.

In [ ]:
"""
================================================================================
FIGURE 2 — QLoRA ADAPTATION BUDGET SWEEP ON CLEAN DATA
================================================================================
Re-runs the sweep that produced the original Figure 2, this time on the
deduplicated held-out set and with an adaptation pool verified disjoint from it.
The original sweep (1.163 -> 1.121 -> 1.120) drew its adaptation pool from the
same stream position as its evaluation text, so it cannot be published.

Budgets: 0 (no adaptation), 1M, 10M tokens.
Scores each on the clean held-out set under both protocols, then plots Protocol B.

RUNTIME on a Kaggle T4, roughly:
    tokenize an 11M-token pool      ~15 min
    train 1M   (488 steps)          ~19 min
    train 10M  (4,882 steps)       ~190 min
    scoring, 3 models x 2 protocols ~20 min
                                   --------
                                   ~4 hours

RESTARTABLE. Each budget saves its adapter and its result. Re-running skips
anything already finished, so a session timeout costs you only the budget that
was in flight. If you have to split it across sessions, commit the notebook
("Save & Run All") so /kaggle/working persists.

REQUIREMENTS
    Run 02_clean_eval_and_score.py in this session first (or have its outputs in
    /kaggle/working): uz_heldout_dedup.txt and dedup_chunks_from_scratch.csv.
    !pip install -q transformers datasets peft bitsandbytes accelerate matplotlib
    Accelerator: GPU T4. Internet: ON.
================================================================================
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, math, json, csv, time
import numpy as np
import torch

LN2  = math.log(2.0)
WORK = "/kaggle/working"
CLEAN_FILE = f"{WORK}/uz_heldout_dedup.txt"
if not os.path.exists(CLEAN_FILE):
    raise SystemExit(f"missing {CLEAN_FILE} — run 02_clean_eval_and_score.py in this session first")
TEXT = open(CLEAN_FILE, encoding="utf-8").read()

BASE      = "ai-forever/mGPT"
BUDGETS   = [0, 1_000_000, 10_000_000]
CTX_TRAIN = 512
PER_DEV, ACC = 2, 2
LR        = 2e-4
SEED      = 20260913

CHUNK_TOKENS, SPAN_BYTES, CONTEXT_BYTES, MAX_TOKENS = 512, 800, 300, 512
FROM_SCRATCH_B = 1.0281      # from results_v3.json, Protocol B
FROM_SCRATCH_A = 1.0264      # Protocol A

assert torch.cuda.is_available(), "Enable a GPU: Settings -> Accelerator -> GPU T4."
DEV  = "cuda"
BF16 = torch.cuda.get_device_capability(0)[0] >= 8
DT   = torch.bfloat16 if BF16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} | bf16: {BF16}")

def free(): gc.collect(); torch.cuda.empty_cache()

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer, default_data_collator)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from torch.utils.data import Dataset

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.model_max_length = 10**9

# ---------------------- pool, sized for the largest budget -------------------
POOL_FILE = f"{WORK}/sweep_pool_{max(BUDGETS)}.npy"
if os.path.exists(POOL_FILE):
    POOL = np.load(POOL_FILE)
    print(f"reusing pool: {len(POOL)/1e6:.2f}M tokens")
else:
    need = int(max(BUDGETS) * 1.10)
    print(f"tokenizing ~{need/1e6:.1f}M adaptation tokens (streaming; ~15 min)...")
    ds = load_dataset("HuggingFaceFW/fineweb-2", name="uzn_Latn",
                      split="train", streaming=True)
    ids, texts, t0 = [], [], time.time()
    for ex in ds:
        t = (ex.get("text") or "").strip()
        if not t: continue
        texts.append(t)
        ids.extend(tok(t, add_special_tokens=False)["input_ids"] + [tok.eos_token_id])
        if len(ids) >= need: break
        if len(ids) % 2_000_000 < 5000:
            print(f"  {len(ids)/1e6:.1f}M tokens, {(time.time()-t0)/60:.1f} min")
    POOL = np.asarray(ids[:need], dtype=np.int32)
    np.save(POOL_FILE, POOL)
    # the eval set comes from val.bin (end of corpus); the pool streams from the
    # start, so they should be disjoint. Verify at the text level anyway.
    pool_text = "\n".join(texts)
    probes = [TEXT[i:i+300] for i in np.linspace(0, len(TEXT)-400, 12).astype(int)]
    leak = sum(1 for p in probes if p in pool_text)
    print(f"tokenized {len(POOL)/1e6:.2f}M tokens | eval-text probes in pool: {leak}/12")
    if leak:
        raise SystemExit("ABORT — adaptation pool overlaps the eval set.")

class Blocks(Dataset):
    def __init__(s, a, c): s.a, s.c, s.n = a, c, len(a)//c
    def __len__(s): return s.n
    def __getitem__(s, i):
        x = s.a[i*s.c:(i+1)*s.c].astype(np.int64)
        return {"input_ids": torch.from_numpy(x), "labels": torch.from_numpy(x.copy())}

def bnb_cfg():
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=DT, bnb_4bit_use_double_quant=True)

def train_budget(budget):
    adir = f"{WORK}/sweep_adapter_{budget}"
    if os.path.exists(f"{adir}/adapter_model.safetensors"):
        print(f"  reusing adapter for {budget:,}"); return adir
    steps = budget // (PER_DEV * ACC * CTX_TRAIN)
    print(f"  training {budget:,} tokens = {steps:,} steps...")
    m = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb_cfg(),
                                             device_map={"": 0})
    m.config.use_cache = False
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    m = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                                     task_type="CAUSAL_LM", target_modules="all-linear"))
    args = TrainingArguments(
        output_dir=adir, per_device_train_batch_size=PER_DEV,
        gradient_accumulation_steps=ACC, max_steps=steps, learning_rate=LR,
        lr_scheduler_type="cosine", warmup_steps=max(10, steps//20),
        logging_steps=max(50, steps//10), save_strategy="no", report_to=[],
        fp16=not BF16, bf16=BF16, gradient_checkpointing=True,
        optim="paged_adamw_8bit", seed=SEED)
    t0 = time.time()
    Trainer(model=m, args=args, train_dataset=Blocks(POOL[:int(budget*1.05)], CTX_TRAIN),
            data_collator=default_data_collator).train()
    print(f"  trained in {(time.time()-t0)/60:.1f} min")
    m.save_pretrained(adir); del m; free()
    return adir

def logits_of(model, x):
    out = model(x)
    return out[0] if isinstance(out, tuple) else getattr(out, "logits", out)

@torch.no_grad()
def protocol_a(model):
    ids = tok(TEXT, add_special_tokens=False)["input_ids"]
    n = len(ids)//CHUNK_TOKENS
    C = torch.tensor(ids[:n*CHUNK_TOKENS], dtype=torch.long).view(n, CHUNK_TOKENS)
    nats, byts = 0.0, 0
    for i in range(n):
        x = C[i:i+1].to(DEV)
        lg = logits_of(model, x); sl, st = lg[:, :-1, :].float(), x[:, 1:]
        nats += torch.nn.functional.cross_entropy(
            sl.reshape(-1, sl.size(-1)), st.reshape(-1), reduction="sum").item()
        byts += len(tok.decode(st[0].tolist()).encode("utf-8"))
    return nats/(LN2*byts)

def build_spans(text, span_bytes):
    spans, i, n, k = [], 0, len(text), 0
    while i < n:
        j, b = i, 0
        while j < n and b < span_bytes:
            b += len(text[j].encode("utf-8")); j += 1
        spans.append((f"span_{k:05d}", i, j)); i, k = j, k+1
    if spans and len(text[spans[-1][1]:spans[-1][2]].encode("utf-8")) < span_bytes//2:
        spans.pop()
    return spans

SPANS = build_spans(TEXT, SPAN_BYTES)

@torch.no_grad()
def protocol_b(model, tag):
    rows = []
    for span_id, a, b in SPANS:
        n_bytes = len(TEXT[a:b].encode("utf-8"))
        c, got = a, 0
        while c > 0 and got < CONTEXT_BYTES:
            c -= 1; got += len(TEXT[c].encode("utf-8"))
        enc = tok(TEXT[c:b], return_offsets_mapping=True, add_special_tokens=False)
        ids, offs = enc["input_ids"], enc["offset_mapping"]
        first = next((k for k,(s,e) in enumerate(offs) if s >= a-c), len(ids))
        if len(ids) < 2: continue
        if len(ids) > MAX_TOKENS:
            drop = len(ids)-MAX_TOKENS; ids = ids[drop:]; first = max(0, first-drop)
        x = torch.tensor(ids, dtype=torch.long, device=DEV).unsqueeze(0)
        lg = logits_of(model, x)
        logits = lg[:, :-1, :].float(); target = x[:, 1:].clone()
        if first > 1: target[:, :first-1] = -100
        rows.append((span_id, torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)), target.reshape(-1),
            ignore_index=-100, reduction="sum").item(), n_bytes))
    with open(f"{WORK}/sweep_chunks_{tag}.csv","w",newline="",encoding="utf-8") as fh:
        w = csv.writer(fh); w.writerow(["chunk_id","sum_nats","n_bytes"])
        w.writerows([(i, f"{v:.6f}", n) for i,v,n in rows])
    return sum(r[1] for r in rows)/(LN2*sum(r[2] for r in rows))

# ------------------------------- sweep --------------------------------------
RES_FILE = f"{WORK}/figure2_clean.json"
RES = json.load(open(RES_FILE)) if os.path.exists(RES_FILE) else {}

for budget in BUDGETS:
    key = str(budget)
    if key in RES:
        print(f"\n[{budget:,}] cached: B={RES[key]['bpb_b']:.4f}"); continue
    print(f"\n{'='*60}\nbudget {budget:,} tokens\n{'='*60}")
    adir = train_budget(budget) if budget > 0 else None
    m = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb_cfg(),
                                             device_map={"": 0}).eval()
    if adir: m = PeftModel.from_pretrained(m, adir).eval()
    bb = protocol_b(m, key); ba = protocol_a(m)
    RES[key] = {"budget": budget, "bpb_b": bb, "bpb_a": ba}
    json.dump(RES, open(RES_FILE,"w"), indent=2)
    print(f"  Protocol B: {bb:.4f} | Protocol A: {ba:.4f}")
    del m; free()

# ------------------------------- plot ---------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

xs = [RES[str(b)]["budget"] for b in BUDGETS]
ys = [RES[str(b)]["bpb_b"] for b in BUDGETS]
xpos = list(range(len(xs)))

fig, ax = plt.subplots(figsize=(7.0, 4.4), dpi=200)
ax.plot(xpos, ys, "o-", color="#1f4e79", lw=2, ms=7, label="mGPT-1.3B + QLoRA", zorder=3)
for x, y in zip(xpos, ys):
    ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, 11),
                ha="center", fontsize=9, color="#1f4e79")
ax.axhline(FROM_SCRATCH_B, ls="--", color="#c0392b", lw=1.6,
           label=f"uzbek-gpt-103m (from scratch), {FROM_SCRATCH_B:.4f}", zorder=2)
ax.set_xticks(xpos)
ax.set_xticklabels(["0\n(zero-shot)", "1M", "10M"])
ax.set_xlabel("Uzbek adaptation tokens")
ax.set_ylabel("Bits-per-byte  (lower is better)")
ax.set_title("Adaptation budget vs. bits-per-byte, deduplicated held-out set",
             fontsize=11, pad=12)
lo = min(min(ys), FROM_SCRATCH_B); hi = max(max(ys), FROM_SCRATCH_B); pad = (hi-lo)*0.28 or 0.01
ax.set_ylim(lo-pad, hi+pad)
ax.grid(axis="y", alpha=0.25, ls=":")
ax.legend(frameon=False, fontsize=9, loc="center right")
for s in ("top","right"): ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(f"{WORK}/figure2_clean.png", bbox_inches="tight")
print(f"\nsaved -> {WORK}/figure2_clean.png")

# --------------------------- caption + prose --------------------------------
d_1m   = ys[1] - ys[0]
d_10m  = ys[2] - ys[1]
gap_10 = ys[2] - FROM_SCRATCH_B

print(f"""
{'='*72}
RESULTS
{'='*72}
  0 tokens   Protocol B {RES['0']['bpb_b']:.4f}   Protocol A {RES['0']['bpb_a']:.4f}
  1M tokens  Protocol B {RES[str(BUDGETS[1])]['bpb_b']:.4f}   Protocol A {RES[str(BUDGETS[1])]['bpb_a']:.4f}
  10M tokens Protocol B {RES[str(BUDGETS[2])]['bpb_b']:.4f}   Protocol A {RES[str(BUDGETS[2])]['bpb_a']:.4f}

  0 -> 1M   change {d_1m:+.4f} bpb
  1M -> 10M change {d_10m:+.4f} bpb   (tenfold data increase)
  gap to from-scratch at 10M: {gap_10:+.4f} bpb

DRAFT CAPTION (check the numbers read correctly before using):

  Figure 2. Bits-per-byte of the mGPT-1.3B + QLoRA baseline as a function of
  adaptation tokens (0, 1M, 10M), evaluated on the deduplicated held-out set
  under Protocol B. The adaptation pool was verified disjoint from the
  evaluation text. A tenfold increase in adaptation data (1M to 10M) changes
  bits-per-byte by {d_10m:.4f}, leaving the adapted baseline {gap_10:.3f} above the
  from-scratch model's {FROM_SCRATCH_B:.4f} (dashed line). Created by the author.
{'='*72}
""")


GPU: Tesla T4 | bf16: False
tokenizing ~11.0M adaptation tokens (streaming; ~15 min)...
  0.0M tokens, 0.0 min
  2.0M tokens, 0.1 min
  4.0M tokens, 0.2 min
  6.0M tokens, 0.3 min
  8.0M tokens, 0.3 min
  10.0M tokens, 0.4 min
tokenized 11.00M tokens | eval-text probes in pool: 0/12

budget 0 tokens

  Protocol B: 1.0774 | Protocol A: 1.1091

budget 1,000,000 tokens
  training 1,000,000 tokens = 488 steps...

 [488/488 16:40]
Step	Training Loss
50	2.150048
100	2.098822
150	2.083660
200	2.060147
250	2.022373
300	2.071900
350	2.055871
400	2.067715
450	2.015471
  trained in 16.7 min

  Protocol B: 1.0757 | Protocol A: 1.1024

budget 10,000,000 tokens
  training 10,000,000 tokens = 4,882 steps...

 [4882/4882 2:47:09]
Step	Training Loss
488	2.021007
976	1.990334
1464	2.017603
1952	1.993641
2440	1.971481
2928	1.954848
3416	1.971256
3904	1.968203
4392	1.945687
4880	1.962652
  trained in 167.2 min

  Protocol B: 1.0591 | Protocol A: 1.0840

saved -> /kaggle/working/figure2_clean.png

RESULTS


## Result

| Adaptation tokens | 0 | 1M | 10M |
|---|---|---|---|
| Bits/byte (Protocol B) | 1.0774 | 1.0757 | 1.0591 |

- Adaptation does **not** plateau: a tenfold increase (1M → 10M) gains 0.0166 bits/byte.
- At 10M it is still **0.031** behind uzbek-gpt-103m (1.0281).
- A rough log-linear extrapolation from the last two points puts the budget needed to close the gap near 700M tokens, the same order as the from-scratch model's own corpus. Two points is thin evidence; read it as an order of magnitude.
- The 1M point here (1.0757) trains on a slightly different slice of the same documents than part A's baseline (1.0764–1.0766), which is why it differs slightly.

The figure is saved as `/kaggle/working/figure2_clean.png`.